In [1]:
import tkinter as tk
from tkinter import messagebox, simpledialog
import sqlite3
from datetime import datetime

# --- Database setup ---
conn = sqlite3.connect("blog.db")
c = conn.cursor()
c.execute('''
CREATE TABLE IF NOT EXISTS posts (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    body TEXT NOT NULL,
    created TEXT NOT NULL
)
''')
conn.commit()

# --- Functions ---
def load_posts():
    post_list.delete(0, tk.END)
    c.execute("SELECT id, title, created FROM posts ORDER BY id DESC")
    for post in c.fetchall():
        post_list.insert(tk.END, f"{post[0]}. {post[1]} ({post[2]})")

def show_post(event):
    if not post_list.curselection():
        return
    selected = post_list.get(post_list.curselection()[0])
    post_id = selected.split('.')[0]
    c.execute("SELECT title, body FROM posts WHERE id = ?", (post_id,))
    post = c.fetchone()
    if post:
        post_text.delete("1.0", tk.END)
        post_text.insert(tk.END, f"Title: {post[0]}\n\n{post[1]}")

def create_post():
    title = simpledialog.askstring("New Post", "Enter title:")
    if not title:
        return
    body = simpledialog.askstring("New Post", "Enter body:")
    if not body:
        return
    created = datetime.now().strftime("%Y-%m-%d %H:%M")
    c.execute("INSERT INTO posts (title, body, created) VALUES (?, ?, ?)", (title, body, created))
    conn.commit()
    load_posts()

def delete_post():
    if not post_list.curselection():
        messagebox.showerror("Error", "No post selected to delete.")
        return
    selected = post_list.get(post_list.curselection()[0])
    post_id = selected.split('.')[0]
    confirm = messagebox.askyesno("Confirm Delete", "Are you sure you want to delete this post?")
    if confirm:
        c.execute("DELETE FROM posts WHERE id = ?", (post_id,))
        conn.commit()
        load_posts()
        post_text.delete("1.0", tk.END)

def edit_post():
    if not post_list.curselection():
        messagebox.showerror("Error", "No post selected to edit.")
        return
    selected = post_list.get(post_list.curselection()[0])
    post_id = selected.split('.')[0]
    c.execute("SELECT title, body FROM posts WHERE id = ?", (post_id,))
    post = c.fetchone()
    if post:
        new_title = simpledialog.askstring("Edit Post", "Edit title:", initialvalue=post[0])
        new_body = simpledialog.askstring("Edit Post", "Edit body:", initialvalue=post[1])
        if new_title and new_body:
            c.execute("UPDATE posts SET title = ?, body = ? WHERE id = ?", (new_title, new_body, post_id))
            conn.commit()
            load_posts()

# --- GUI Setup ---
root = tk.Tk()
root.title("Tkinter Blog - One Window")
root.geometry("800x500")
root.configure(bg="#f0f0f0")

# Left Frame: Post list
left_frame = tk.Frame(root)
left_frame.pack(side=tk.LEFT, fill=tk.Y, padx=10, pady=10)

tk.Label(left_frame, text="Posts", font=("Helvetica", 14)).pack()
post_list = tk.Listbox(left_frame, width=40, height=20)
post_list.pack(padx=5, pady=5)
post_list.bind("<<ListboxSelect>>", show_post)

# Right Frame: Post content
right_frame = tk.Frame(root)
right_frame.pack(side=tk.RIGHT, expand=True, fill=tk.BOTH, padx=10, pady=10)

tk.Label(right_frame, text="Post Content", font=("Helvetica", 14)).pack()
post_text = tk.Text(right_frame, wrap=tk.WORD, font=("Arial", 12))
post_text.pack(expand=True, fill=tk.BOTH)

# Bottom Buttons
btn_frame = tk.Frame(root, bg="#f0f0f0")
btn_frame.pack(side=tk.BOTTOM, fill=tk.X, pady=10)

tk.Button(btn_frame, text="New Post", width=15, command=create_post).pack(side=tk.LEFT, padx=10)
tk.Button(btn_frame, text="Edit Post", width=15, command=edit_post).pack(side=tk.LEFT, padx=10)
tk.Button(btn_frame, text="Delete Post", width=15, command=delete_post).pack(side=tk.LEFT, padx=10)

# Load posts on startup
load_posts()

# Run the app
root.mainloop()
